# 02 缺失功率模型与验证

目标：理解为什么缺失功率不能直接填 0，以及为什么要做分组验证。

边界：这个模型只在“已报告功率的 TOP500 机器”上验证；它不能自动推出全球 HPC。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, r2_score

BASE = Path.cwd()
if BASE.name == 'notebooks':
    BASE = BASE.parent
DATA = BASE.parent / '2026-09-17 第一次课' / '重写论文' / 'analysis' / 'inputs' / 'top500_full.csv'
print(DATA)

top500 = pd.read_csv(DATA)
print(top500.shape)
top500.head()

In [ ]:
reported = top500[top500['Power (kW)'].notna()].copy()
missing = top500[top500['Power (kW)'].isna()].copy()
print('Rows:', len(top500))
print('Reported power:', len(reported))
print('Missing power:', len(missing))
print('Reported power sum kW:', f"{reported['Power (kW)'].sum():,.2f}")

In [ ]:
reported['has_accelerator'] = reported['Accelerator/Co-Processor'].notna().astype(float)
X = np.column_stack([
    np.log10(reported['Rmax [TFlop/s]'].to_numpy()),
    reported['has_accelerator'].to_numpy(),
    (reported['Year'].to_numpy() - 2020) / 10,
])
y = np.log10(reported['Power (kW)'].to_numpy())
groups = reported['Site ID'].fillna(reported['Name']).to_numpy()

pred = np.zeros_like(y)
for train_idx, test_idx in GroupKFold(n_splits=5).split(X, y, groups):
    model = Ridge(alpha=1.0)
    model.fit(X[train_idx], y[train_idx])
    pred[test_idx] = model.predict(X[test_idx])

rmse = mean_squared_error(y, pred) ** 0.5
r2 = r2_score(y, pred)
mult_error = np.median(10 ** np.abs(y - pred))
print(f'log10 RMSE: {rmse:.3f}')
print(f'R2: {r2:.3f}')
print(f'Median multiplicative error: {mult_error:.2f}x')

In [ ]:
import matplotlib.pyplot as plt

observed_kw = 10 ** y
predicted_kw = 10 ** pred

plt.figure(figsize=(6, 5))
plt.scatter(observed_kw, predicted_kw, alpha=0.75)
plt.xscale('log')
plt.yscale('log')
lo = min(observed_kw.min(), predicted_kw.min())
hi = max(observed_kw.max(), predicted_kw.max())
plt.plot([lo, hi], [lo, hi], '--', color='gray')
plt.xlabel('Observed power (kW)')
plt.ylabel('Site-held-out prediction (kW)')
plt.title('Power model validation')
plt.show()

解释这张图时，不要只说“模型还不错”。要说明：

- 每个点是一台已报告功率的机器
- 横轴是真实功率，纵轴是被留出站点的预测功率
- 离对角线越远，预测误差越大
- 验证不能证明未报告机器一定服从同一规律

In [ ]:
# 用全部已报告功率训练，再对缺失机器做条件预测。
# 这只是辅助情景，不作为主结论。
fit = Ridge(alpha=1.0).fit(X, y)
missing = missing.copy()
missing['has_accelerator'] = missing['Accelerator/Co-Processor'].notna().astype(float)
Xm = np.column_stack([
    np.log10(missing['Rmax [TFlop/s]'].to_numpy()),
    missing['has_accelerator'].to_numpy(),
    (missing['Year'].to_numpy() - 2020) / 10,
])
missing['predicted_power_kW'] = 10 ** fit.predict(Xm)
print('Predicted missing-power sum kW:', f"{missing['predicted_power_kW'].sum():,.0f}")
missing[['Rank', 'Name', 'Country', 'predicted_power_kW']].head()